# 데이터 ZIP 검증 (수집·학습 아님)

ImageNet/NAIP ZIP을 이미 준비한 뒤 실행하는 읽기 전용 검사입니다. 이 notebook은 원저자 영상이나 NAIP 영상2000장을 자동 수집하지 않습니다. ImageNet은 이용 권한이 있는 공식 데이터를 직접 준비하고, NAIP 대체 데이터는 Google Earth Engine 공식 catalog에서 직접 수집해야 합니다. 수집 규칙과 원본 영상 목록이 미공개이므로 저자 데이터와 동일하다고 주장하지 않습니다.

입력: `imagenet_train_10000.zip` 10000장, `GEE_train_fixed.zip` 1000장, `GEE_test.zip` 1000장. ImageNet 전체 TAR는 Colab/Drive에 올릴 필요가 없으며, 저장소의 `tools/prepare_imagenet_subset.py --source ... --output ...`로 subset만 준비할 수 있습니다.

검사는 PNG/JPEG 개수·중복 이름·파일SHA·train/test exact-file 겹침을 확인합니다. 공간적 인접·내용상 유사성·훈련 checkpoint의 실제 데이터 이력을 보증하지 않습니다. 동일파일 기준으로 test 중복이나 학습/test 겹침이 있으면 중단합니다. ZIP과 이미지는 변경하지 않습니다.

출처: [ImageNet](https://www.image-net.org/download.php), [Earth Engine NAIP catalog](https://developers.google.com/earth-engine/datasets/catalog/USDA_NAIP_DOQQ). 필요한 프로젝트 등록·데이터 접근 권한은 사용자가 준비해야 하며 이 notebook에 계정/API key를 넣지 마세요.


In [ ]:
from pathlib import Path
import hashlib
import json
import zipfile
from google.colab import drive

drive.mount("/content/drive")
DATA_DIR=Path("/content/drive/MyDrive/FSO_data")
expected={"imagenet_train_10000.zip":10000,"GEE_train_fixed.zip":1000,"GEE_test.zip":1000}
inventories={}
zip_hashes={}
for filename,count in expected.items():
    path=DATA_DIR/filename
    if not path.is_file():
        raise FileNotFoundError(f"Prepare the licensed/proxy data first: {filename}")
    digest=hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda:stream.read(8*1024**2),b""):
            digest.update(block)
    zip_hashes[filename]=digest.hexdigest()
    images={}
    with zipfile.ZipFile(path) as archive:
        for item in archive.infolist():
            if item.is_dir() or Path(item.filename).suffix.lower() not in {".png",".jpg",".jpeg"}:
                continue
            if item.filename in images:
                raise ValueError("Duplicate image member name in "+filename)
            images[item.filename]=hashlib.sha256(archive.read(item)).hexdigest()
    if len(images)!=count:
        raise ValueError(f"{filename}:expected{count},found{len(images)}")
    inventories[filename]=images
    print(filename,len(images),"images; distinct exact files",len(set(images.values())),flush=True)
test_hashes=set(inventories["GEE_test.zip"].values())
if len(test_hashes)!=1000:
    raise ValueError("Duplicate exact files in test pool; do not silently discard them")
for filename in ("imagenet_train_10000.zip","GEE_train_fixed.zip"):
    if test_hashes.intersection(inventories[filename].values()):
        raise ValueError("Training/test exact-file overlap: "+filename)
report={"images":expected,"zip_sha256":zip_hashes,"test_train_exact_file_overlap":0,
        "scope":"exact file identities only; not geographic independence or checkpoint provenance"}
print(json.dumps(report,indent=2))
print("ZIP contents were not changed. Next:02_train.ipynb")
